In [34]:
import csv
import jieba
import pickle

# 用户评论数据集
ds_comments = []
with open('short_comments.csv', 'r') as file:
    reader = csv.DictReader(file)
    for row in reader:
        vote = int(row['Star'])
        words = jieba.lcut(row['Comment'])
        if(vote ==1 or vote ==2):
            ds_comments.append((words, 0))
        elif(vote ==4 or vote ==5):
            ds_comments.append((words, 1))

len(ds_comments)

# 保存
with open('ds_comments.pkl', 'wb') as file:
    pickle.dump(ds_comments, file)

In [35]:
import pickle
import torch.nn as nn
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence  # 长度不同张量填充为相同长度
import jieba

#定义一个将分词结果转换为 {word: index}的函数
def build_vocab(doc):
    vocab = set()
    for line in doc:
        vocab.update(line[0])
    vocab =  ['PAD','UNK'] + list(vocab)
    w_index = {word: idx for idx, word in enumerate(vocab)}
    return w_index



#1、加载评论分词完成的数据
comments_data = []
with open('ds_comments.pkl','rb') as f:
    comments_data = pickle.load(f)
    # print(len(comments_data)) # 130行评论数据

#2、构建词表
vocab = build_vocab(comments_data)
# print(len(vocab)) #1862个词

#3、词表变成词向量
# emb = nn.Embedding(len(vocab), 100,0)
# print(emb) #Embedding(1862, 100, padding_idx=0)

#4、将词向量，每个batch的维度变成一致的，需要使用collate_fn回调函数处理
def convert_data(batch_data):
        comments, votes = [],[]
        # 分别提取评论和标签
        for comment, vote in batch_data:
            comments.append(torch.tensor([vocab.get(word, vocab['UNK']) for word in comment]))
            votes.append(vote)
        
        # 将评论和标签转换为tensor
        data_x = pad_sequence(comments, batch_first=True, padding_value=vocab['PAD'])  # 统一为相同长度
        data_y = torch.tensor(votes)
        # 返回评论和标签
        return data_x, data_y
    
dataloader = DataLoader(dataset = comments_data, batch_size=5, shuffle=True, 
                            collate_fn=convert_data)
# print(len(dataloader)) # 33个batch
print(type(dataloader))  # 确认类型


<class 'torch.utils.data.dataloader.DataLoader'>


In [36]:
# 5、构建 RNN 模型
class Comments_Classifier(nn.Module):
 
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)  # padding_idx=0
        self.rnn = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
 
    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        output, (hidden, _) = self.rnn(embedded)
        output = self.fc(output[:, -1, :]) 
        return output
#6、定义模型参数
# vocab_size: 词汇表大小
# embedding_dim: 词嵌入维度
# hidden_size: LSTM隐藏层大小
# num_classes: 分类数量
vocab_size = len(vocab)
embedding_dim = 100
hidden_size = 128
num_classes = 2
learning_rate= 0.01
num_epochs = 50

In [37]:
#7、开始训练
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Comments_Classifier(len(vocab), embedding_dim, hidden_size, num_classes)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for epoch in range(num_epochs):
    for i, (inputs, labels) in enumerate(dataloader):
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if (i+1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            
# 保存模型
torch.save(model.state_dict(), 'comments_classifier.pth')
# 模型词典
torch.save(vocab, 'comments_vocab.pth')

Epoch [1/50], Loss: 0.4787
Epoch [1/50], Loss: 1.0569
Epoch [2/50], Loss: 0.9146
Epoch [2/50], Loss: 0.6564
Epoch [3/50], Loss: 0.6961
Epoch [3/50], Loss: 0.5887
Epoch [4/50], Loss: 0.6241
Epoch [4/50], Loss: 0.6098
Epoch [5/50], Loss: 0.5982
Epoch [5/50], Loss: 0.7649
Epoch [6/50], Loss: 0.3929
Epoch [6/50], Loss: 0.5616
Epoch [7/50], Loss: 0.5665
Epoch [7/50], Loss: 0.5557
Epoch [8/50], Loss: 0.4327
Epoch [8/50], Loss: 0.5401
Epoch [9/50], Loss: 0.9018
Epoch [9/50], Loss: 0.5336
Epoch [10/50], Loss: 0.8788
Epoch [10/50], Loss: 0.6329
Epoch [11/50], Loss: 0.2057
Epoch [11/50], Loss: 0.6830
Epoch [12/50], Loss: 0.6488
Epoch [12/50], Loss: 0.5367
Epoch [13/50], Loss: 0.6546
Epoch [13/50], Loss: 0.6479
Epoch [14/50], Loss: 0.4375
Epoch [14/50], Loss: 0.2620
Epoch [15/50], Loss: 0.6988
Epoch [15/50], Loss: 0.4371
Epoch [16/50], Loss: 0.1404
Epoch [16/50], Loss: 0.0750
Epoch [17/50], Loss: 0.5903
Epoch [17/50], Loss: 0.0847
Epoch [18/50], Loss: 0.0691
Epoch [18/50], Loss: 0.0314
Epoch [19/

In [46]:
# -------模型预测---------
embedding_dim = 100
hidden_size = 128
num_classes = 2

# 加载词典
vocab = torch.load('comments_vocab.pth')
# 测试模型
comment1 = '我很少评价电影，除非它真的难入眼'
comment2 = '二刷二刷，情节牛逼'

# 将评论转换为索引
comment1_idx = torch.tensor([vocab.get(word, vocab['UNK']) for word in jieba.lcut(comment1)])
comment2_idx = torch.tensor([vocab.get(word, vocab['UNK']) for word in jieba.lcut(comment2)])
# 将评论转换为tensor
comment1_idx = comment1_idx.unsqueeze(0).to(device)  # 添加batch维度    
comment2_idx = comment2_idx.unsqueeze(0).to(device)  # 添加batch维度

# 加载模型
model = Comments_Classifier(len(vocab), embedding_dim, hidden_size, num_classes)
model.load_state_dict(torch.load('comments_classifier.pth'))
model.to(device)

# 模型推理
pred1 = model(comment1_idx)
pred2 = model(comment2_idx)

# 取最大值的索引作为预测结果
pred1 = torch.argmax(pred1, dim=1).item()
pred2 = torch.argmax(pred2, dim=1).item()
print(f'评论1预测结果: {pred1}')
print(f'评论2预测结果: {pred2}')


评论1预测结果: 0
评论2预测结果: 1
